# Clase 7 · Notebook 02
# Transfer learning: reutilizar una red ya entrenada

**Introducción al Deep Learning — Módulo 3, Unidad 2: Redes de neuronas residuales**

El **transfer learning** consiste en aprovechar lo que una red ya aprendió (la **extracción de características**)
para resolver una tarea nueva, entrenando solo una **cabeza de clasificación** nueva.

Para que el cuaderno funcione sin descargas, lo demostramos con un caso pequeño y controlado:

1. Entrenamos una **red base** para reconocer los dígitos **0–4**.
2. **Congelamos** su extractor de características.
3. Le añadimos una **cabeza nueva** y la entrenamos para los dígitos **5–9** (tarea nueva).
4. Al final, mostramos el código real de transfer learning con **ResNet50** (como referencia).

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Dos tareas a partir del dataset de dígitos

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
tf.random.set_seed(42); np.random.seed(42)

digits = load_digits()
X = digits.images.reshape(-1, 8, 8, 1).astype("float32") / 16.0
y = digits.target

# Tarea A (base): dígitos 0-4 | Tarea B (nueva): dígitos 5-9
maskA, maskB = y < 5, y >= 5
XA, yA = X[maskA], y[maskA]
XB, yB = X[maskB], y[maskB] - 5      # reetiquetamos 5-9 como 0-4

yA = tf.keras.utils.to_categorical(yA, 5)
yB = tf.keras.utils.to_categorical(yB, 5)
XA_tr, XA_te, yA_tr, yA_te = train_test_split(XA, yA, test_size=0.2, random_state=42)
XB_tr, XB_te, yB_tr, yB_te = train_test_split(XB, yB, test_size=0.2, random_state=42)
print("Tarea A (0-4):", XA.shape[0], "imágenes | Tarea B (5-9):", XB.shape[0], "imágenes")

## 2. Entrenar la red base (tarea A: dígitos 0-4)

In [ ]:
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input

# Extractor de características (lo que reutilizaremos)
entrada = Input(shape=(8, 8, 1))
e = Conv2D(16, (3, 3), padding="same", activation="relu")(entrada)
e = MaxPooling2D((2, 2))(e)
e = Conv2D(32, (3, 3), padding="same", activation="relu")(e)
e = Flatten()(e)
extractor = Model(entrada, e, name="extractor")

# Red base = extractor + cabeza A
base = Sequential([extractor, Dense(32, activation="relu"), Dense(5, activation="softmax")])
base.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
base.fit(XA_tr, yA_tr, epochs=15, batch_size=32, verbose=0)
print("Red base (0-4) -> accuracy: %.1f %%" % (base.evaluate(XA_te, yA_te, verbose=0)[1] * 100))

## 3. Transferir: congelar el extractor y entrenar una cabeza nueva

Marcamos `extractor.trainable = False`: sus pesos **no se reentrenan**. Solo entrenamos la cabeza nueva
para la tarea B (dígitos 5-9). Así reutilizamos las características ya aprendidas.

In [ ]:
extractor.trainable = False     # congelamos el extractor

modelo_transfer = Sequential([
    extractor,
    Dense(32, activation="relu"),
    Dense(5, activation="softmax"),
])
modelo_transfer.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
modelo_transfer.fit(XB_tr, yB_tr, epochs=15, batch_size=32, verbose=0)

acc_transfer = modelo_transfer.evaluate(XB_te, yB_te, verbose=0)[1]
print("Transfer learning (5-9, extractor congelado) -> accuracy: %.1f %%" % (acc_transfer * 100))
print("\nReutilizamos las características de la tarea A para resolver la tarea B,")
print("entrenando MUCHOS menos parámetros (solo la cabeza nueva).")

## 4. En la práctica: transfer learning con ResNet50

En problemas reales se reutiliza una red potente **preentrenada con millones de imágenes** (ImageNet).
Este es el patrón que vimos en teoría con **ResNet50** (este código necesita conexión para descargar los
pesos, por eso lo mostramos como referencia):

```python
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Flatten, Dense, Dropout

# 1) Base preentrenada SIN la cabeza de clasificación, con los pesos de ImageNet
base = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base.trainable = False                       # congelamos la extracción

# 2) Nuestra cabeza de clasificación (3 clases: caballo / perro / gato)
x = GlobalAveragePooling2D()(base.output)
x = Flatten()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.6)(x)
salida = Dense(3, activation="softmax")(x)

modelo = Model(base.input, salida)
modelo.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
```

## Resumen

- **Transfer learning**: reutilizar la **extracción** de una red entrenada y entrenar solo una **cabeza nueva**.
- Se **congela** la base (`trainable = False`) para no perder lo aprendido.
- Ahorra tiempo, datos y cómputo; ideal cuando tenemos pocos datos.
- Con `ResNet50(weights="imagenet", include_top=False)` se hace en pocas líneas.

Has completado la unidad de redes residuales. En la próxima clase veremos las **GANs**.